## Feature Engineering  

## 1. Objetivo

Criar variáveis que ajudam o modelo a entender o tempo

## 2 Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sqlite3

from IPython.display import Markdown
from IPython.core.display import HTML
import math

import glob
import warnings
warnings.filterwarnings('ignore')

## 3. Configuraçãões Iniciais

In [ ]:
# Cor principal do projeto
PRIMARY_COLOR = "#50e550"
SECONDARY_COLORS = sns.light_palette(PRIMARY_COLOR, n_colors=5)

# Estilo geral
sns.set_theme(style="whitegrid")

# Tamanho padrão
plt.rcParams['figure.figsize'] = (10, 6)

# Fonte
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

## 4. Carregamento e Leitura dos Dados


In [ ]:
df_demand = pd.read_parquet('../data/processed/df_demand.parquet')

df_demand.head()

## 5. Preparação inicial

In [ ]:
df = df_demand.copy()

df['date'] = pd.to_datetime(df['date'])

df = df.sort_values(['product_id', 'date'])

## 6. Features de tempo

Capturam padrões como sazonalidade semanal e mensal.

In [ ]:
df['day_of_week'] = df['date'].dt.dayofweek
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['week_of_year'] = df['date'].dt.isocalendar().week

## 7. Features de Lag

Capturam dependência temporal (o passado influencia o futuro).

In [ ]:
df['lag_1'] = df.groupby('product_id')['sales'].shift(1)
df['lag_7'] = df.groupby('product_id')['sales'].shift(7)
df['lag_14'] = df.groupby('product_id')['sales'].shift(14)

## 8. Médias Móveis

Suavizam a série e ajudam a capturar tendência.

In [ ]:
df['rolling_mean_7'] = (
    df.groupby('product_id')['sales']
      .shift(1)
      .rolling(7)
      .mean()
)

df['rolling_mean_14'] = (
    df.groupby('product_id')['sales']
      .shift(1)
      .rolling(14)
      .mean()
)

In [ ]:
df['rolling_std_7'] = (
    df.groupby('product_id')['sales']
      .shift(1)
      .rolling(7)
      .std()
)

## 9. Tratamento de valores faltantes (gerados pelas features)

Tratamento de NaNs

As features de lag e rolling geram valores faltantes no início da série.
Esses registros são removidos para garantir consistência na modelagem.

In [ ]:
df_features = df.dropna()

In [ ]:

df_features.info()

## 10. Salvar dataset

In [ ]:
df_features.to_parquet('../data/processed/df_features.parquet')

## 11. Conclusão

Foram criadas features relevantes que capturam padrões temporais e comportamento histórico da demanda.

Essas variáveis são fundamentais para modelos de previsão, pois permitem:
- capturar sazonalidade
- entender dependência temporal
- reduzir ruído da série

O dataset final será utilizado na etapa de modelagem.